# Tier 1 Latency — Writeup

Autonomous run, 2026-08-01, branch `revival/tier1-latency`. Full narrative,
every measured number, and every self-critique live in
`TIER1_PROGRESS_LOG.md` (append-only) — this notebook is a compact summary,
not a replacement for that ledger. Where a number below is loaded from a
real captured file, the cell shows it; where a number is transcribed from a
ledger entry (because it came from a live AWS call made earlier in the run,
not reproducible for free here), that is stated explicitly.


## Step 0 — the measurement that gated everything

Real data, loaded from disk below (not transcribed).

In [1]:
import polars as pl
from pathlib import Path

df = pl.read_parquet(Path.cwd() / "step0_results_20260801_130201.parquet")
df.select(["query", "run_idx", "variant_gen_ms", "s3_query_ms", "wall_ms"])


query,run_idx,variant_gen_ms,s3_query_ms,wall_ms
str,i64,f64,f64,f64
"""What was NVIDIA's revenue in 2…",0,2322.751292,3118.063125,5909.58525
"""What was NVIDIA's revenue in 2…",1,1965.891333,1551.186875,3814.382416
"""What was NVIDIA's revenue in 2…",2,1663.757167,1258.984084,3191.136667
"""How did Microsoft describe clo…",0,1672.666333,2452.654792,4349.230709
"""How did Microsoft describe clo…",1,1847.65175,1947.935167,4094.835708
…,…,…,…,…
"""What were Apple's R&D expenses…",1,1986.265458,1084.737292,3346.751458
"""What were Apple's R&D expenses…",2,2051.619208,1379.088458,3658.702667
"""Describe Tesla's risk factors …",0,1994.3985,3047.203708,5316.69975


In [2]:
summary = df.select(
    pl.col("variant_gen_ms").median().alias("variant_gen_median_ms"),
    pl.col("s3_query_ms").median().alias("s3_query_median_ms"),
    pl.col("wall_ms").median().alias("wall_median_ms"),
)
summary


variant_gen_median_ms,s3_query_median_ms,wall_median_ms
f64,f64,f64
1990.331979,1465.137666,3794.5025


**Finding:** `variant_gen_median_ms` > `s3_query_median_ms`. The pre-existing
claim in `CLAUDE.md` ("retrieve (S3 Vectors) is ~90% of the time") is false in
its attribution — the larger cost inside that block is one Haiku rephrase call
plus embeddings, not S3 Vectors querying itself.

**Gate result:** S (median `s3_query_ms`) sits just under the 1500ms
abandon threshold set in `TIER1_LATENCY_DESIGN.md` *before* this measurement
was taken. Change 3 (concurrent retrieval) was abandoned per that
pre-committed rule. The margin is thin (~35ms on a 12-run sample) and that is
stated plainly in the ledger, not overstated as a clean result.

## Change 1 — component caching (task #19)

Real data, loaded from disk below.

In [3]:
import re

log_path = Path.cwd().parent / "TIER1_PROGRESS_LOG.md"
log_text = log_path.read_text()

# Pull the three measured PASS lines out of the real log entry rather than
# retyping the numbers by hand.
pass_lines = re.findall(r"PASS \d+:\s+([\d.]+) ms", log_text)
pl.DataFrame({
    "pass": ["PASS 1 (cold build)", "PASS 2 (cached)", "PASS 3 (cached)"],
    "ms": [float(x) for x in pass_lines[:3]],
})


pass,ms
str,f64
"""PASS 1 (cold build)""",1164.7
"""PASS 2 (cached)""",0.0
"""PASS 3 (cached)""",0.0


Extracted directly from the ledger text above (not retyped) — cold build
pays the full construction cost once per process; every call after is a
dict/attribute lookup.

## Changes 2, 4a, 4b, and VERIFY — transcribed from the ledger

These numbers came from live AWS/Bedrock calls and a real Docker container
made earlier in this run. They are not re-derived here (that would mean
spending real money again just to populate a notebook) — they are
transcribed from `TIER1_PROGRESS_LOG.md`, and that file remains the primary
source of truth for each one.

In [4]:
summary_rows = [
    # (change, metric, value, verified_how)
    ("Change 2 (event loop)", "/health during a 10.1s query", "1.5-9.6 ms (3 probes)", "real concurrent HTTP calls"),
    ("Change 2 (event loop)", "2 concurrent queries, wall time", "9.1s / 9.7s (not ~18s serial)", "real concurrent HTTP calls"),
    ("Change 2 (event loop)", "log rows under concurrency", "2/2 landed, 0 lost, 161ms apart", "real S3 query_logs.parquet read"),
    ("Change 4a (progress events)", "stages fired, in order", "5/5 (entities,embed,retrieve,expand,assemble)", "real supply-line-2 call + callback"),
    ("Change 4b (streaming)", "SSE events, one real query", "5 stage + 95 token + 1 replace + 1 done", "curl against local backend"),
    ("Change 4b (streaming, browser)", "citation-chip render, metadata", "claude-haiku-4-5, 12706 tok, $0.0129, 6.9s", "real Streamlit browser test"),
    ("VERIFY (Docker)", "TTFB vs total (proves no uvicorn buffering)", "0.0043s vs 8.96s (~2000x apart)", "curl -w against real container"),
    ("VERIFY (Docker)", "browser test through real container", "META FY21 citation chip, $0.0151, 8.2s", "real Streamlit browser test, containerized"),
    ("VERIFY (Docker)", "backend memory after 3 real queries", "1.257 GiB (of a 3072 MiB ECS task)", "docker stats"),
]
pl.DataFrame(summary_rows, schema=["change", "metric", "value", "verified_how"], orient="row")


change,metric,value,verified_how
str,str,str,str
"""Change 2 (event loop)""","""/health during a 10.1s query""","""1.5-9.6 ms (3 probes)""","""real concurrent HTTP calls"""
"""Change 2 (event loop)""","""2 concurrent queries, wall tim…","""9.1s / 9.7s (not ~18s serial)""","""real concurrent HTTP calls"""
"""Change 2 (event loop)""","""log rows under concurrency""","""2/2 landed, 0 lost, 161ms apar…","""real S3 query_logs.parquet rea…"
"""Change 4a (progress events)""","""stages fired, in order""","""5/5 (entities,embed,retrieve,e…","""real supply-line-2 call + call…"
"""Change 4b (streaming)""","""SSE events, one real query""","""5 stage + 95 token + 1 replace…","""curl against local backend"""
"""Change 4b (streaming, browser)""","""citation-chip render, metadata""","""claude-haiku-4-5, 12706 tok, $…","""real Streamlit browser test"""
"""VERIFY (Docker)""","""TTFB vs total (proves no uvico…","""0.0043s vs 8.96s (~2000x apart…","""curl -w against real container"""
"""VERIFY (Docker)""","""browser test through real cont…","""META FY21 citation chip, $0.01…","""real Streamlit browser test, c…"
"""VERIFY (Docker)""","""backend memory after 3 real qu…","""1.257 GiB (of a 3072 MiB ECS t…","""docker stats"""


## What did NOT ship, and why — this is not a hidden list

- **Change 3 (concurrent retrieval):** abandoned per the Step 0 gate. See
  above.
- **Byte-identical streaming-vs-non-streaming answer parity:** not
  meaningful to test — `semantic_variants.temperature: 0.7` already makes
  repeat calls to the same query nondeterministic, so an exact-match test
  would measure the LLM's sampling, not this change.
- **N-way concurrent SSE streams:** only ever tested one stream at a time.
- **Cancellation on client disconnect:** the streaming worker thread has no
  cancellation signal; a closed browser tab does not stop the pipeline or
  its cost.
- **Container memory re-sizing:** 1.257 GiB observed after only 3 queries in
  one session, against a 3072 MiB ECS task allocation — flagged as worth
  re-checking under sustained load, not resized (out of the 3 approved
  changes' scope).

Two further items are open for Joel's judgment in `TIER1_OPEN_QUESTIONS.md`:
warm-vs-lazy component init on ECS, and whether to pursue overlapping
variant generation with base retrieval — a narrower, lower-risk version of
the abandoned Change 3, motivated by Step 0's own finding that variant
generation, not S3 Vectors, is the larger cost.

## Bottom line

Five of the eight queue items shipped and were independently verified
against real AWS this run (Step 0, Change 1, Change 2, Change 4a, Change 4b),
one was gated and honestly abandoned with the reasoning preserved (Change 3),
and end-to-end Docker verification confirmed all of it survives contact with
a real container, not just a bare Python process. Every commit lives on
`revival/tier1-latency`; `main` was never touched.